# What does cross-validation actually decide?

`RidgeClassifierCV` picks its regularization strength by
cross-validation. We trace one fit and read that decision as two
checkable inequalities. Every claim below is followed by the numbers
that back it.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, "../skverify-hypothesis")

import numpy as np
import sympy
from sklearn.linear_model import RidgeClassifierCV
from skverify import latex
from skverify_hypothesis import explore, edge_cases, verify

def fit_coef(X, yraw):
    y = (yraw > 0.0).astype(float)
    return RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(X, y).coef_.ravel()

rng = np.random.default_rng(0)
X = rng.standard_normal((8, 2)); yraw = rng.standard_normal(8)

sklearn_says = fit_coef(X.copy(), yraw.copy())
out = verify(fit_coef, X, yraw)     # traces AND checks the values match
print("sklearn coefficients:    ", np.round(sklearn_says, 6))
print("certificate coefficients:", np.round(np.asarray(out.value, dtype=float).ravel(), 6))

sklearn coefficients:     [0.037205 0.153496]
certificate coefficients: [0.037205 0.153496]


Same numbers. `verify` already asserted that; the print is for your
eyes. Now the interesting part: which alpha did the model pick, and
can we see why?

In [2]:
clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(X, (yraw > 0).astype(float))
print("sklearn picked alpha =", clf.alpha_)

sklearn picked alpha = 10.0


In [3]:
from IPython.display import Math, display
alpha_guards = [g for g in out.preconditions.args if "solve_eigen" in str(g)]
legend = {}
for g in alpha_guards:
    display(Math(latex(g, aliases=legend)))
for short, full in legend.items():
    print(f"{short} = {full}")
print("(three residual terms, one per alpha, in order: 0.1, 1.0, 10.0)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

T1 = solve_eigen_covariance_4_0
T2 = solve_eigen_covariance_5_0
T3 = solve_eigen_covariance_6_0
(three residual terms, one per alpha, in order: 0.1, 1.0, 10.0)


Read them as sentences: the mean squared cross-validation residual
of one alpha is at least that of another. The legend names the terms;
they belong to alpha 0.1, 1.0 and 10.0 in call order. Check the claim
with sklearn's own numbers:

In [4]:
clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0], store_cv_results=True).fit(
    X, (yraw > 0).astype(float))
preds = clf.cv_results_[:, 0, :]
Y = np.where((yraw > 0), 1.0, -1.0)
err = ((preds - Y[:, None]) ** 2).mean(0)
for a, e in zip([0.1, 1.0, 10.0], err):
    mark = "  <- winner" if a == clf.alpha_ else ""
    print(f"alpha {a:>5}: mean squared LOO error {e:8.4f}{mark}")

alpha   0.1: mean squared LOO error  22.4508
alpha   1.0: mean squared LOO error   8.0282
alpha  10.0: mean squared LOO error   3.3180  <- winner


How many different computations can this one estimator run? Thirty
random datasets, one witness per distinct set of conditions:

In [5]:
paths = explore(fit_coef, (np.zeros((8, 2)), np.zeros(8)), max_examples=30)
alphas_seen = {}
for args_, o in paths:
    Xa, ya = args_
    a = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(Xa, (ya > 0).astype(float)).alpha_
    alphas_seen[a] = alphas_seen.get(a, 0) + 1
print(len(paths), "distinct computational paths in 30 draws")
print("alpha chosen across those paths:", dict(sorted(alphas_seen.items())))

30 distinct computational paths in 30 draws
alpha chosen across those paths: {0.1: 2, 1.0: 4, 10.0: 24}


Every path is a different branch: a different label split or a
different alpha winner. Last, the fragile inputs: samples sitting
exactly on the class threshold. Watch the coefficients move as one
sample crosses from one class to the tie:

In [6]:
base_X = paths[0][0][0].copy(); base_y = paths[0][0][1].copy()
sample = 0
for delta, label in [(0.5, "clearly class 1"), (1e-9, "just above"), (0.0, "exactly on the threshold")]:
    y_mod = base_y.copy(); y_mod[sample] = delta
    c = fit_coef(base_X.copy(), y_mod)
    print(f"yraw[0] = {delta:<12} ({label:24s}) -> coef {np.round(c, 4)}")

yraw[0] = 0.5          (clearly class 1         ) -> coef [-0.0859 -0.2361]
yraw[0] = 1e-09        (just above              ) -> coef [-0.0859 -0.2361]
yraw[0] = 0.0          (exactly on the threshold) -> coef [-0.0917 -0.2093]


At `0.0` the sample lands in class 0 (the threshold is strict), and
the coefficients jump: that is the branch boundary the certificate's
condition `yraw[0] > 0` named. `edge_cases` builds these inputs
automatically for every condition in the certificate:

In [7]:
cases = edge_cases(fit_coef, (np.zeros((8, 2)), np.zeros(8)), paths=paths[:3])
print(len(cases), "boundary inputs built, one per condition")

14 boundary inputs built, one per condition


**Takeaway.** The traced fit matches sklearn to the last digit. The
alpha choice is two inequalities you can read and check. Thirty random
datasets gave twenty-odd distinct branches, and every branch boundary
comes with an input that sits exactly on it. No sklearn source was
read to learn any of this.